# 14 · Uncertainty-tracking lab on real CMU-MOSEI (CPU, minutes)

Runs every method in `lab/methods.py` on the cache written by notebook 13 (attach its output). Each method may
learn only from the self-generated corruptions (white noise λ = 1, random swap) in train / valid; the test set adds
held-out corruptions (noise λ = 0.5, half-clip noise, gain fault, dead sensor, time shuffle, nearest-neighbour swap,
two noisy modalities). Scores: NLL, coverage gap of 90% ranges, tracking ratio (σ rise / error rise, ideal 1) and
AUROC of σ for corrupted vs clean clips.

In [ ]:
# Setup: make the `fusionlab` library importable. On Kaggle, always use a FRESH clone of the latest code (kept in /tmp,
# so it is not saved into this notebook's output); attached copies are used only when there is no Internet.
import glob, os, subprocess, sys

REPO_URL = 'https://github.com/fasihh/bigthing'      # the JF/ folder holds this project
BRANCH = 'jf-experiments'                           # used only until JF/ is merged into main


def fresh_clone(dest='/tmp/bigthing'):
    if not os.path.isdir(os.path.join(dest, '.git')):
        r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, dest], capture_output=True, text=True)
        if r.returncode:
            return None
    if not os.path.isdir(os.path.join(dest, 'JF')):  # JF not merged into main yet: use its branch
        subprocess.run(['git', '-C', dest, 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', dest, 'checkout', '-q', 'FETCH_HEAD'], check=True)
    return os.path.join(dest, 'JF')


def find_jf():
    here = os.path.abspath('.')
    if not here.startswith('/kaggle'):                # running inside a local checkout of the repo
        for c in (os.path.dirname(here), here, os.path.join(here, 'JF')):
            if os.path.isdir(os.path.join(c, 'src', 'fusionlab')):
                return c
    jf = fresh_clone()
    if jf:
        return jf
    for c in glob.glob('/kaggle/input/**/JF', recursive=True):   # offline fallback (may be older code)
        if os.path.isdir(os.path.join(c, 'src', 'fusionlab')):
            print('WARNING: no Internet, using an attached copy of the code:', c)
            return c
    raise RuntimeError('Cannot find the fusionlab code: turn on Internet in the notebook settings.')


JF = find_jf()
sys.path.insert(0, os.path.join(JF, 'src'))
CONFIGS = os.path.join(JF, 'configs')
commit = subprocess.run(['git', '-C', JF, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('using fusionlab from', JF, '| commit', commit or 'n/a')

In [ ]:
import glob, subprocess
sys.path.insert(0, JF)
cache = glob.glob('/kaggle/input/**/uq_cache.npz', recursive=True)[0]
print(cache)
r = subprocess.run([sys.executable, '-m', 'lab.run', 'real', '--cache', cache, '--tags', 'vjepa_frozen', 'vjepa_vt_frozen',
                    'vjepa_finetuned', '--out', '/kaggle/working/lab_real.csv'], cwd=JF, capture_output=True, text=True)
print(r.stdout[-20000:]); print(r.stderr[-5000:])

In [ ]:
import pandas as pd
from lab.run import summary
df = pd.read_csv('/kaggle/working/lab_real.csv')
pd.set_option('display.width', 250)
for tag in df.tag.unique():
    print('=====', tag); print(summary(df[df.tag == tag]).round(3).to_string())

In [ ]:
# per held-out kind, NLL
print(df[df.heldout].groupby(['tag', 'kind', 'method']).NLL.mean().unstack('kind').round(3).to_string())